# Preprocessing Debug

This notebook verifies Task 8: cyclic padding, log-mel conversion, and per-sample normalization.

In [2]:
from pathlib import Path
import sys

import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import preprocess_cycle

with open(PROJECT_ROOT / "configs" / "baseline.yaml", "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

train_df = pd.read_csv(PROJECT_ROOT / "data" / "splits" / "train.csv")
raw_dir = PROJECT_ROOT / "data" / "raw"


def sample_row_for_label(label: int):
    return train_df[train_df["label"] == label].iloc[0]

d:\younes\respiratory-classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
for label in [0, 1, 2, 3]:
    row = sample_row_for_label(label)
    wav_path = raw_dir / f"{row['recording_file']}.wav"
    spec = preprocess_cycle(str(wav_path), float(row['start']), float(row['end']), config)
    print(f"label={label} shape={spec.shape} mean={spec.mean():.3f} std={spec.std():.3f} nan={bool((spec != spec).any())}")
    assert spec.ndim == 2
    assert spec.shape[0] == 128
    assert abs(float(spec.mean())) < 0.1
    assert not bool((spec != spec).any())

print("✅ Preprocessing verified.")

label=0 shape=(128, 345) mean=-0.000 std=1.000 nan=False
label=1 shape=(128, 345) mean=-0.000 std=1.000 nan=False
label=2 shape=(128, 345) mean=-0.000 std=1.000 nan=False
label=3 shape=(128, 345) mean=-0.000 std=1.000 nan=False
✅ Preprocessing verified.
